# 03 - MLP em PyTorch para Churn

Etapa 2: Construção, treinamento e avaliação de uma Rede Neural Multicamadas (MLP) usando PyTorch para previsão de churn.

**Sumário:**
1. Setup e bibliotecas
2. Preparação dos dados
3. Definição da arquitetura MLP
4. Treinamento e avaliação
5. Discussão dos resultados

## 1) Setup e bibliotecas

Importação das principais bibliotecas utilizadas:
- PyTorch para modelagem e treinamento
- Numpy e pandas para manipulação de dados
- Matplotlib para visualização
- Configuração do device (GPU/CPU)

In [30]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device em uso: {device}')

Device em uso: cpu


## 2) Preparação dos dados
Reaproveitamento do pipeline de preprocessamento do baseline para garantir comparabilidade entre modelos.

- Carrega os splits já tratados (X_train, X_test, y_train, y_test) exportados do notebook baseline.
- Converte todos os dados para arrays float32, garantindo compatibilidade com PyTorch.
- Exporta os dados como tensores PyTorch prontos para uso no modelo.

In [31]:
# Carrega dados já tratados do baseline (ajuste o caminho se necessário)
from joblib import load
from pathlib import Path

# Corrige o caminho para o arquivo de splits exportado do baseline:
splits_path = Path('data/processed/baseline_splits_arrays.joblib')  # Caminho relativo à raiz do projeto
X_train, X_test, y_train, y_test = load(splits_path)

# Diagnóstico dos tipos e formatos dos dados carregados
print("X_train:", type(X_train), getattr(X_train, 'shape', None))
print("X_test:", type(X_test), getattr(X_test, 'shape', None))
print("y_train:", type(y_train), getattr(y_train, 'shape', None))
print("y_test:", type(y_test), getattr(y_test, 'shape', None))

# Se forem DataFrames ou Series, mostre as primeiras linhas e dtypes
if hasattr(X_train, 'head'):
    print("X_train dtypes:\n", X_train.dtypes)
    display(X_train.head())
if hasattr(y_train, 'head'):
    print("y_train dtypes:\n", y_train.dtypes)
    display(y_train.head())

X_train: <class 'numpy.ndarray'> (39239, 101)
X_test: <class 'numpy.ndarray'> (9810, 101)
y_train: <class 'pandas.core.series.Series'> (39239,)
y_test: <class 'pandas.core.series.Series'> (9810,)
y_train dtypes:
 int64


43741    0
32439    0
36388    0
38518    1
15710    0
Name: churn, dtype: int64

In [32]:
# Após o diagnóstico, converte os dados para float32 caso necessário e cria tensores PyTorch
def to_float32_array(arr):
    if hasattr(arr, 'to_numpy'):
        return arr.to_numpy(dtype=np.float32)
    return np.asarray(arr, dtype=np.float32)

X_train_np = to_float32_array(X_train)
X_test_np = to_float32_array(X_test)
y_train_np = to_float32_array(y_train)
y_test_np = to_float32_array(y_test)

X_train_tensor = torch.tensor(X_train_np, dtype=torch.float32).to(device)
X_test_tensor = torch.tensor(X_test_np, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train_np, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test_np, dtype=torch.float32).to(device)

print('Shapes:', X_train_tensor.shape, X_test_tensor.shape)

Shapes: torch.Size([39239, 101]) torch.Size([9810, 101])


## 3) Definição da arquitetura MLP
Criação de uma rede neural multicamadas (MLP) simples para classificação binária de churn.

- Duas camadas ocultas com ReLU para capturar não-linearidades.
- Camada de saída com ativação sigmoid para prever probabilidade de churn.
- O summary do modelo é exibido para conferência da arquitetura.

In [33]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden1=64, hidden2=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

# Instancia o modelo
input_dim = X_train_tensor.shape[1]
model = MLP(input_dim).to(device)
print(model)

MLP(
  (net): Sequential(
    (0): Linear(in_features=101, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
    (5): Sigmoid()
  )
)


## 4) Treinamento e avaliação do MLP
Nesta etapa, o modelo MLP é treinado com os dados de treino e avaliado no conjunto de teste.

- Utiliza-se a função de perda BCELoss (binária) e otimizador Adam.
- O loop de treinamento mostra a evolução da loss.
- Após o treinamento, calcula-se a acurácia no conjunto de teste para avaliação preliminar.

In [34]:
## 4) Treinamento do modelo MLP
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

n_epochs = 20
for epoch in range(n_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor).squeeze()
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1}/{n_epochs}, Loss: {loss.item():.4f}')

# 5) Avaliação preliminar no conjunto de teste
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor).squeeze()
    preds = (test_outputs > 0.5).cpu().numpy()
    y_true = y_test_tensor.cpu().numpy()
    acc = (preds == y_true).mean()
    print(f'Acurácia no teste: {acc:.4f}')

Epoch 1/20, Loss: 0.6946
Epoch 5/20, Loss: 0.6802
Epoch 10/20, Loss: 0.6607
Epoch 15/20, Loss: 0.6355
Epoch 20/20, Loss: 0.6023
Acurácia no teste: 0.7089
